# Jersey Pipeline (Colab + Resume + Drive Persistence)

Use this notebook for repeatable runs on ephemeral Colab runtimes:

- Keep **repo / dataset zip / weights / outputs** on Drive.
- Run in `/content` (fast local disk) every session.
- Use `--resume` (default) so completed pipeline stages are skipped.
- Use `--force` only when you need a full recompute.

**Changing runtime (CPU ↔ GPU) or reconnecting starts a new VM** — `/content` is wiped. **Re-run all cells from the top** (Drive mount → sync repo → dataset → weights → install) before `main.py`.

**Dataset:** Set `DATASET_MODE = 'split_zips'` if you use `train.zip` / `test.zip`. Zips are auto-found in `MyDrive/<project>/<repo>/data/SoccerNet/jersey-2023/` or `MyDrive/<project>/data/...` or project root. Override with `DRIVE_SPLIT_ZIPS_DIR` if needed. Set `CACHE_EXTRACTED_DATASET_TO_DRIVE = True` once to persist extracted `jersey-2023` under `MyDrive/<project>/data/SoccerNet/jersey-2023/`. Use `FORCE_REBUILD_DRIVE_DATASET = True` to replace a bad extract.


In [ ]:
# ===== CONFIG =====
DRIVE_PROJECT = 'jersey-number-pipeline'   # folder under MyDrive
REPO_NAME = 'jersey-number-pipeline'
REPO_URL = 'https://github.com/superbolt08/jersey-number-pipeline.git'

# Dataset on Drive — set DATASET_MODE below
DATASET_ZIP_NAME = 'jersey-2023.zip'   # single combined zip (DATASET_MODE == 'single_zip')
DATASET_MODE = 'single_zip'           # 'single_zip' | 'split_zips' (train.zip, test.zip, ...)
DRIVE_SPLIT_ZIPS = ['train.zip', 'test.zip', 'challenge.zip']  # any subset; missing files skipped
FORCE_REBUILD_DRIVE_DATASET = False   # True = delete Drive cache before re-extract (fix bad partial)
# Optional: absolute Colab path to folder containing train.zip / test.zip. '' = auto (see mount cell).
DRIVE_SPLIT_ZIPS_DIR = ''

# Weights folder layout on Drive:
# MyDrive/<DRIVE_PROJECT>/weights/models/*
# MyDrive/<DRIVE_PROJECT>/weights/reid/*
# MyDrive/<DRIVE_PROJECT>/weights/pose/*
WEIGHTS_DIR_NAME = 'weights'

# Pipeline run behavior
PART = 'test'               # test / val / challenge
RESUME = True               # True -> --resume
FORCE = False               # True -> --force (overrides resume)

# Keep this False for full tracklet run (recommended for final evaluation)
LIMIT_TRACKLETS = False

# If True, after extracting to /content, rsync jersey-2023 -> Drive (reuse next session without unzip)
CACHE_EXTRACTED_DATASET_TO_DRIVE = False


In [ ]:
from google.colab import drive
import os, shutil, subprocess, textwrap, pathlib, sys

drive.mount('/content/drive')

def run(cmd, cwd=None):
    """Run shell command; capture merged stdout+stderr so failures show the real traceback."""
    print(f'\n[RUN] {cmd}')
    p = subprocess.run(
        cmd, shell=True, cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if p.stdout:
        print(p.stdout, end='')
    if p.returncode != 0:
        raise RuntimeError(f'Command failed ({p.returncode}): {cmd}')

# Google Drive over FUSE: rsync often exits 23 ("partial transfer") on unreadable shortcuts or temp files.
# Drop --delete here: we rm -rf LOCAL_REPO_DIR first; --delete adds Drive metadata churn.
_RSYNC_FROM_DRIVE = (
    'rsync -rltD --no-perms --no-owner --no-group --modify-window=2 '
    '--copy-links --partial '
    '--exclude=".tmp.drivedownload" --exclude=".Trash*" '
)

def run_rsync_from_drive(src_dir, dst_dir):
    cmd = f'{_RSYNC_FROM_DRIVE} "{src_dir}/" "{dst_dir}/"'
    print(f'\n[RUN] {cmd}')
    p = subprocess.run(cmd, shell=True)
    if p.returncode == 0:
        return
    if p.returncode == 23:
        print('WARN: rsync exit 23 (partial transfer). Common with Drive; if clone looks OK, continue.')
        print('Tip: remove non-file Google shortcuts from the repo folder on Drive, or retry after remount.')
        return
    raise RuntimeError(f"rsync failed ({p.returncode}): {cmd}")

DRIVE_ROOT = '/content/drive/MyDrive'
DRIVE_PROJECT_DIR = os.path.join(DRIVE_ROOT, DRIVE_PROJECT)
DRIVE_REPO_DIR = os.path.join(DRIVE_PROJECT_DIR, REPO_NAME)
DRIVE_DATASET_ZIP = os.path.join(DRIVE_PROJECT_DIR, DATASET_ZIP_NAME)
DRIVE_WEIGHTS_DIR = os.path.join(DRIVE_PROJECT_DIR, WEIGHTS_DIR_NAME)
DRIVE_DATASET_EXTRACTED = os.path.join(DRIVE_PROJECT_DIR, 'data', 'SoccerNet', 'jersey-2023')

# Where train.zip / test.zip live (auto: .../<REPO>/data/SoccerNet/jersey-2023 next to your repo clone)
if DRIVE_SPLIT_ZIPS_DIR:
    DRIVE_SPLIT_ZIPS_ROOT = DRIVE_SPLIT_ZIPS_DIR
else:
    _zip_candidates = [
        os.path.join(DRIVE_REPO_DIR, 'data', 'SoccerNet', 'jersey-2023'),
        os.path.join(DRIVE_PROJECT_DIR, 'data', 'SoccerNet', 'jersey-2023'),
        DRIVE_PROJECT_DIR,
    ]
    DRIVE_SPLIT_ZIPS_ROOT = DRIVE_PROJECT_DIR
    for _zc in _zip_candidates:
        if any(os.path.isfile(os.path.join(_zc, z)) for z in DRIVE_SPLIT_ZIPS):
            DRIVE_SPLIT_ZIPS_ROOT = _zc
            break
print('Folder for split zips (train.zip, test.zip, …):', DRIVE_SPLIT_ZIPS_ROOT)

LOCAL_REPO_DIR = os.path.join('/content', REPO_NAME)
LOCAL_DATA_DIR = os.path.join(LOCAL_REPO_DIR, 'data', 'SoccerNet')
LOCAL_DATASET_ROOT = os.path.join(LOCAL_DATA_DIR, 'jersey-2023')

os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print('Drive project dir:', DRIVE_PROJECT_DIR)
print('Local repo dir:', LOCAL_REPO_DIR)


In [ ]:
# 1) Ensure repo exists on Drive (persistent), then sync to fast local disk (/content)
if not os.path.isdir(os.path.join(DRIVE_REPO_DIR, '.git')):
    run(f'git clone {REPO_URL} "{DRIVE_REPO_DIR}"')
else:
    run('git pull', cwd=DRIVE_REPO_DIR)

if os.path.isdir(LOCAL_REPO_DIR):
    run(f'rm -rf "{LOCAL_REPO_DIR}"')
run_rsync_from_drive(DRIVE_REPO_DIR, LOCAL_REPO_DIR)

# Clone required sub-repos if missing (into local working copy)
os.makedirs(os.path.join(LOCAL_REPO_DIR, 'reid'), exist_ok=True)
os.makedirs(os.path.join(LOCAL_REPO_DIR, 'pose'), exist_ok=True)
os.makedirs(os.path.join(LOCAL_REPO_DIR, 'str'), exist_ok=True)

# SAM: empty sam2/ from Drive still counts as "exists" — require sam.py or re-clone
_sam2 = os.path.join(LOCAL_REPO_DIR, 'sam2')
if not os.path.isfile(os.path.join(_sam2, 'sam.py')):
    if os.path.isdir(_sam2):
        run(f'rm -rf "{_sam2}"')
    run('git clone --recurse-submodules https://github.com/davda54/sam.git sam2', cwd=LOCAL_REPO_DIR)
_reid = os.path.join(LOCAL_REPO_DIR, 'reid', 'centroids-reid')
if not os.path.isfile(os.path.join(_reid, 'train_ctl_model.py')):
    if os.path.isdir(_reid):
        run(f'rm -rf "{_reid}"')
    run('git clone --recurse-submodules https://github.com/mikwieczorek/centroids-reid.git reid/centroids-reid', cwd=LOCAL_REPO_DIR)
# PyTorch Lightning 2.x: upstream centroids-reid imports seed_everything from a removed path
_misc = os.path.join(LOCAL_REPO_DIR, 'reid', 'centroids-reid', 'utils', 'misc.py')
if os.path.isfile(_misc):
    with open(_misc, encoding='utf-8') as _f:
        _t = _f.read()
    _needle = 'from pytorch_lightning.utilities.seed import seed_everything'
    if _needle in _t and 'lightning_fabric.utilities.seed' not in _t:
        _t = _t.replace(
            _needle,
            'try:\n    from pytorch_lightning.utilities.seed import seed_everything\n'
            'except ImportError:\n    from lightning_fabric.utilities.seed import seed_everything',
        )
        with open(_misc, 'w', encoding='utf-8') as _f:
            _f.write(_t)
        print('Patched centroids-reid utils/misc.py for PyTorch Lightning 2.x')
if not os.path.isdir(os.path.join(LOCAL_REPO_DIR, 'pose', 'ViTPose')):
    run('git clone --recurse-submodules https://github.com/ViTAE-Transformer/ViTPose.git pose/ViTPose', cwd=LOCAL_REPO_DIR)
# ViTPose mmpose caps mmcv at 1.5.0; torch2 prebuilts are mmcv-full 1.7.x — bump cap before pip install.
_patch_vit = os.path.join(LOCAL_REPO_DIR, 'scripts', 'patch_vitpose_mmpose_mmcv_range.py')
_vit_root = os.path.join(LOCAL_REPO_DIR, 'pose', 'ViTPose')
if os.path.isfile(_patch_vit):
    run(f'python "{_patch_vit}" "{_vit_root}"', cwd=LOCAL_REPO_DIR)
else:
    _vmi = os.path.join(_vit_root, 'mmpose', '__init__.py')
    if os.path.isfile(_vmi):
        with open(_vmi, encoding='utf-8') as _f:
            _vx = _f.read()
        if "mmcv_maximum_version = '1.5.0'" in _vx:
            _vx = _vx.replace("mmcv_maximum_version = '1.5.0'", "mmcv_maximum_version = '1.7.2'")
            with open(_vmi, 'w', encoding='utf-8') as _f:
                _f.write(_vx)
            print('Patched ViTPose mmpose mmcv cap (fallback).')
if not os.path.isdir(os.path.join(LOCAL_REPO_DIR, 'str', 'parseq')):
    run('git clone --recurse-submodules https://github.com/baudm/parseq.git str/parseq', cwd=LOCAL_REPO_DIR)

print('Repo + sub-repos ready.')

In [ ]:
# 2) Dataset staging (resume-friendly)
# CONFIG: DATASET_MODE 'single_zip' | 'split_zips'; CACHE_EXTRACTED_DATASET_TO_DRIVE; FORCE_REBUILD_DRIVE_DATASET
# split_zips: copies train.zip / test.zip / … from DRIVE_SPLIT_ZIPS_ROOT -> /content, unzips, builds jersey-2023 locally,
#   then rsyncs to Drive when CACHE_EXTRACTED_DATASET_TO_DRIVE is True (persistent for future sessions).
# Next runs: rsync from DRIVE_DATASET_EXTRACTED -> local (skip unzip) if cache exists.

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)


def dataset_ready(root):
    return os.path.isdir(os.path.join(root, 'test', 'images'))


def _normalize_nested_split(jersey_root):
    """Flatten jersey-2023/{split}/{split}/... when SoccerNet uses double nesting."""
    for split in ('train', 'test', 'challenge'):
        nested = os.path.join(jersey_root, split, split)
        if not os.path.isdir(nested):
            continue
        parent = os.path.join(jersey_root, split)
        for name in os.listdir(nested):
            if name.startswith('.'):
                continue
            src = os.path.join(nested, name)
            dst = os.path.join(parent, name)
            if os.path.exists(dst):
                if os.path.isdir(dst):
                    shutil.rmtree(dst)
                else:
                    os.remove(dst)
            shutil.move(src, dst)
        try:
            os.rmdir(nested)
        except OSError:
            pass


def _merge_one_unzipped_piece(jersey_root, piece_dir, zip_name):
    """Merge one extracted archive tree into jersey_root."""
    if not os.path.isdir(piece_dir):
        return
    for split in ('train', 'test', 'challenge'):
        src = os.path.join(piece_dir, split)
        if os.path.isdir(src):
            dst = os.path.join(jersey_root, split)
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            shutil.move(src, dst)
            return
    stem = os.path.splitext(zip_name)[0].lower()
    if stem not in ('train', 'test', 'challenge'):
        stem = 'test'
    img = os.path.join(piece_dir, 'images')
    if os.path.isdir(img):
        dst = os.path.join(jersey_root, stem, 'images')
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.move(img, dst)
        return
    raise RuntimeError(
        'Cannot map %r into jersey layout. Top-level: %s'
        % (zip_name, sorted(os.listdir(piece_dir))[:40])
    )


def _extract_split_zips_from_drive():
    split_zips_found = [
        z for z in DRIVE_SPLIT_ZIPS
        if os.path.isfile(os.path.join(DRIVE_SPLIT_ZIPS_ROOT, z))
    ]
    if not split_zips_found:
        raise FileNotFoundError(
            'No split zips under %s (expected any of %s)'
            % (DRIVE_SPLIT_ZIPS_ROOT, DRIVE_SPLIT_ZIPS)
        )
    stage = '/content/_jersey_split_stage'
    if os.path.isdir(stage):
        shutil.rmtree(stage)
    os.makedirs(stage, exist_ok=True)
    if os.path.isdir(LOCAL_DATASET_ROOT):
        shutil.rmtree(LOCAL_DATASET_ROOT)
    os.makedirs(LOCAL_DATASET_ROOT, exist_ok=True)

    for zname in split_zips_found:
        zpath = os.path.join(DRIVE_SPLIT_ZIPS_ROOT, zname)
        piece = os.path.join(stage, '_piece')
        if os.path.isdir(piece):
            shutil.rmtree(piece)
        os.makedirs(piece, exist_ok=True)
        local_zip = os.path.join('/content', zname)
        run(f'cp "{zpath}" "{local_zip}"')
        run(f'unzip -o -q "{local_zip}" -d "{piece}"')
        _merge_one_unzipped_piece(LOCAL_DATASET_ROOT, piece, zname)
        shutil.rmtree(piece, ignore_errors=True)

    shutil.rmtree(stage, ignore_errors=True)
    _normalize_nested_split(LOCAL_DATASET_ROOT)


if FORCE_REBUILD_DRIVE_DATASET and os.path.isdir(DRIVE_DATASET_EXTRACTED):
    print('FORCE_REBUILD_DRIVE_DATASET: removing', DRIVE_DATASET_EXTRACTED)
    shutil.rmtree(DRIVE_DATASET_EXTRACTED)

split_mode_ok = (
    DATASET_MODE == 'split_zips'
    and any(os.path.isfile(os.path.join(DRIVE_SPLIT_ZIPS_ROOT, z)) for z in DRIVE_SPLIT_ZIPS)
)

if dataset_ready(LOCAL_DATASET_ROOT):
    print('Dataset already present locally, skipping extraction.')
elif dataset_ready(DRIVE_DATASET_EXTRACTED):
    # Prefer cached extract on Drive so you do not re-unzip every new runtime
    print('Syncing extracted dataset from Drive -> local /content ...')
    os.makedirs(os.path.dirname(LOCAL_DATASET_ROOT), exist_ok=True)
    run(f'rsync -a "{DRIVE_DATASET_EXTRACTED}/" "{LOCAL_DATASET_ROOT}/"')
elif split_mode_ok:
    print('DATASET_MODE=split_zips: unzip from Drive -> /content, merge into jersey-2023 ...')
    _extract_split_zips_from_drive()
    if not dataset_ready(LOCAL_DATASET_ROOT):
        raise RuntimeError(
            'After split extract, missing test/images under %s (check zip paths).' % LOCAL_DATASET_ROOT
        )
    if CACHE_EXTRACTED_DATASET_TO_DRIVE:
        os.makedirs(os.path.dirname(DRIVE_DATASET_EXTRACTED), exist_ok=True)
        run(f'rsync -a "{LOCAL_DATASET_ROOT}/" "{DRIVE_DATASET_EXTRACTED}/"')
        print('Persisted extracted dataset on Drive:', DRIVE_DATASET_EXTRACTED)
elif os.path.isfile(DRIVE_DATASET_ZIP):
    local_zip = os.path.join('/content', DATASET_ZIP_NAME)
    run(f'cp "{DRIVE_DATASET_ZIP}" "{local_zip}"')
    run(f'unzip -o -q "{local_zip}" -d "{LOCAL_DATA_DIR}"')
    train_dir = os.path.join(LOCAL_DATA_DIR, 'train')
    test_dir = os.path.join(LOCAL_DATA_DIR, 'test')
    if os.path.isdir(train_dir) and os.path.isdir(test_dir) and not os.path.isdir(LOCAL_DATASET_ROOT):
        os.makedirs(LOCAL_DATASET_ROOT, exist_ok=True)
        run(f'mv "{train_dir}" "{LOCAL_DATASET_ROOT}/"')
        run(f'mv "{test_dir}" "{LOCAL_DATASET_ROOT}/"')
    if CACHE_EXTRACTED_DATASET_TO_DRIVE:
        os.makedirs(os.path.dirname(DRIVE_DATASET_EXTRACTED), exist_ok=True)
        run(f'rsync -a "{LOCAL_DATASET_ROOT}/" "{DRIVE_DATASET_EXTRACTED}/"')
        print('Persisted extracted dataset on Drive:', DRIVE_DATASET_EXTRACTED)
else:
    raise FileNotFoundError(
        'Dataset not found. Expected one of:\n'
        ' - MODE split_zips + zips in %s: %s\n'
        ' - extracted: %s\n'
        ' - single zip: %s'
        % (DRIVE_SPLIT_ZIPS_ROOT, DRIVE_SPLIT_ZIPS, DRIVE_DATASET_EXTRACTED, DRIVE_DATASET_ZIP)
    )

_normalize_nested_split(LOCAL_DATASET_ROOT)
assert dataset_ready(LOCAL_DATASET_ROOT), f'Dataset layout invalid at {LOCAL_DATASET_ROOT}'
print('Dataset ready at', LOCAL_DATASET_ROOT)
_ti = os.path.join(LOCAL_DATASET_ROOT, 'test', 'images')
_nt = len([d for d in os.listdir(_ti) if os.path.isdir(os.path.join(_ti, d))])
print('Tracklet folders under test/images:', _nt)


In [ ]:
# 3) Copy weights from Drive to local repo paths
models_dst = os.path.join(LOCAL_REPO_DIR, 'models')
reid_dst = os.path.join(LOCAL_REPO_DIR, 'reid', 'centroids-reid', 'models')
pose_dst = os.path.join(LOCAL_REPO_DIR, 'pose', 'ViTPose', 'checkpoints')
for p in [models_dst, reid_dst, pose_dst]:
    os.makedirs(p, exist_ok=True)

# Checkpoints on Drive — first match wins (very common: inside the repo clone on Drive):
#  - .../<project>/weights/{models,reid,pose}
#  - .../<project>/{models,reid,pose}
#  - .../<project>/<repo_name>/models  (same paths as git repo)
def _pick_dir(paths):
    for p in paths:
        if p and os.path.isdir(p):
            return p
    return None

_models_cands = [
    os.path.join(DRIVE_WEIGHTS_DIR, 'models'),
    os.path.join(DRIVE_PROJECT_DIR, 'models'),
    os.path.join(DRIVE_REPO_DIR, 'models'),
]
_reid_cands = [
    os.path.join(DRIVE_WEIGHTS_DIR, 'reid'),
    os.path.join(DRIVE_PROJECT_DIR, 'reid'),
    os.path.join(DRIVE_REPO_DIR, 'reid', 'centroids-reid', 'models'),
]
_pose_cands = [
    os.path.join(DRIVE_WEIGHTS_DIR, 'pose'),
    os.path.join(DRIVE_PROJECT_DIR, 'pose'),
    os.path.join(DRIVE_REPO_DIR, 'pose', 'ViTPose', 'checkpoints'),
]
base_models = _pick_dir(_models_cands)
base_reid = _pick_dir(_reid_cands)
base_pose = _pick_dir(_pose_cands)

if not base_models:
    raise FileNotFoundError(
        'Could not find a models/ folder on Drive. Put PARSeq + legibility files in ONE of:\n'
        + '\n'.join(f'  - {p}' for p in _models_cands)
    )

print('Using Drive models from:', base_models)
if base_reid:
    print('Using Drive reid from:', base_reid)
else:
    print('[WARN] ReID folder not found on Drive; tried:\n  ' + '\n  '.join(_reid_cands))
if base_pose:
    print('Using Drive pose from:', base_pose)
else:
    print('[WARN] Pose checkpoints folder not found; tried:\n  ' + '\n  '.join(_pose_cands))

src_map = [(base_models, models_dst)]
if base_reid:
    src_map.append((base_reid, reid_dst))
if base_pose:
    src_map.append((base_pose, pose_dst))
for src, dst in src_map:
    if os.path.isdir(src):
        run(f'rsync -a "{src}/" "{dst}/"')
    else:
        print(f'[WARN] Missing weights subfolder: {src}')

print('Weights staged.')


In [ ]:
# 4) Install dependencies (idempotent)
# Colab already includes CUDA runtime; install project deps in current runtime.
run('python -m pip install -q --upgrade pip', cwd=LOCAL_REPO_DIR)
run('python -m pip install -q -r requirements.txt', cwd=LOCAL_REPO_DIR)
run('python -m pip install -q gdown yacs pytorch-lightning', cwd=LOCAL_REPO_DIR)

# ViTPose: mmcv-full 1.x. `mim install` alone often fails on Colab Py3.12 — script retries wheel indices + build.
run('python scripts/install_mmcv_full_vitpose.py', cwd=LOCAL_REPO_DIR)

# PARSeq deps (for str.py inference path)
run('python -m pip install -q -r str/parseq/requirements/inference.txt', cwd=LOCAL_REPO_DIR)
run('python -m pip install -q -e str/parseq', cwd=LOCAL_REPO_DIR)

# quick sanity (cell 4 must pass these or pose.py will fail)
run('python - <<"PY"\nimport torch\nprint("torch", torch.__version__, "cuda", torch.cuda.is_available())\nimport mmcv\nprint("mmcv", mmcv.__version__)\nPY', cwd=LOCAL_REPO_DIR)


In [ ]:
# 5) Ensure full-tracklet setting for final runs (optional patch)
cfg_path = os.path.join(LOCAL_REPO_DIR, 'configuration.py')
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg_text = f.read()

if not LIMIT_TRACKLETS:
    new_text = cfg_text.replace('soccer_net_max_tracklets = 5', 'soccer_net_max_tracklets = None')
    if new_text != cfg_text:
        with open(cfg_path, 'w', encoding='utf-8') as f:
            f.write(new_text)
        print('Updated soccer_net_max_tracklets to None for full run.')
    else:
        print('soccer_net_max_tracklets already not set to 5; no change.')
else:
    print('Keeping tracklet limit as configured.')


In [ ]:
# 6) Run pipeline with resume/force controls
# (Inlined subprocess capture so you see the full traceback even if an older `run()` is still in RAM.)
import subprocess as _sp
flags = []
if FORCE:
    flags.append('--force')
elif RESUME:
    flags.append('--resume')

cmd = f"python main.py SoccerNet {PART} {' '.join(flags)}".strip()
print('Running:', cmd)
p = _sp.run(
    cmd, shell=True, cwd=LOCAL_REPO_DIR, text=True,
    stdout=_sp.PIPE, stderr=_sp.STDOUT,
)
if p.stdout:
    print(p.stdout, end='')
if p.returncode != 0:
    raise RuntimeError(f'Command failed ({p.returncode}): {cmd}')


In [ ]:
# 7) Persist outputs + optionally notebook-side code changes back to Drive
local_out = os.path.join(LOCAL_REPO_DIR, 'out')
drive_out = os.path.join(DRIVE_PROJECT_DIR, 'out')
os.makedirs(drive_out, exist_ok=True)
if os.path.isdir(local_out):
    run(f'rsync -a "{local_out}/" "{drive_out}/"')
    print('Outputs synced to Drive:', drive_out)
else:
    print('No out/ folder found.')

# Optional: sync modified repo files back to persistent Drive clone (without deleting)
run(f'rsync -a "{LOCAL_REPO_DIR}/" "{DRIVE_REPO_DIR}/"')
print('Repo synced back to Drive clone.')
